# Open-Ended Review Entity & Sentiment Discovery

This notebook demonstrates a first-pass approach for discovering emergent entities from free-form reviews and ranking them by aggregated sentiment. It is intentionally lightweight and assumption-driven so it can be iterated on quickly. Steps:

1. Load reviews from `reviews.csv`.
2. Extract candidate entities (keyphrases) per review.
3. Estimate sentiment per review and assign it to the entities mentioned there.
4. Merge semantically similar entities via embedding clustering.
5. Aggregate sentiments per merged entity to rank reputations.

Assumptions & notes:
- Reviews are free-form text (can be multilingual). We use keyword extraction (YAKE) instead of language-specific NER to avoid schema assumptions.
- Sentiment is inferred at the review level with a multilingual model and propagated to its entities; per-entity sentiment extraction can be added later.
- Semantic merging uses multilingual sentence embeddings with agglomerative clustering; adjust the distance threshold to merge more or fewer entities.


In [1]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from transformers import pipeline
import yake

pd.set_option('display.max_colwidth', None)


In [2]:
# Load reviews

df = pd.read_csv('reviews.csv', header=None, names=['review_text'])
df['review_id'] = df.index

df.head()


,review_text,review_id
0,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문. 배달 받고 보니 서비스라고 콜라까지 보냈다. 따로 리뷰 요청도 없었는데. 마늘 토핑은 안쪽까지 바삭하니 과자 식감이 신기했다.,0
1,처가집 양념통닭에서 마라치킨 주문. 그냥 달달한 양념 치킨에 마라향 살짝 추가한 정도. 닭고기 모든 부위 식감이 퍽퍽한것 같다.,1
2,성남 51번 버스 타고 가는데 기사가 핸드폰을 보면서 운전한다. 기가 막히다.,2
3,도서관 근처에서 51번 버스를 기다리는데 정거장에 서지 않고 그냥 지나간다. 어이가 없다.,3
4,분당 이마트앞 사거리에는 좌회전 신호를 놓치지 않으려는지 버스들이 전력 질주를 한다. 무섭다. 거기 우회전할때도 길건너는 사람 치지 않을지 두렵다.,4


In [3]:
# Extract candidate entities/keyphrases per review using YAKE + NER + light filtering

kw_unigram = yake.KeywordExtractor(lan='ko', n=1, top=12)
kw_ngram = yake.KeywordExtractor(lan='ko', n=3, top=12)

# Multilingual NER to catch entities YAKE misses (open-ended)
ner_model = pipeline(
    'token-classification',
    model='Davlan/bert-base-multilingual-cased-ner-hrl',
    aggregation_strategy='simple',
)

stopwords = {
    '그리고', '하지만', '또한', '정말', '너무', '그냥', '이거', '저거', '그거',
    '이건', '저건', '그건', '이런', '저런', '그런', '합니다', '했어요', '했다',
    '합니다', '있다', '없다', '아니다', '되다', '것', '수', '좀', '더',
    '많이', '조금', '진짜', '완전', '별로', '되게', '아주', '매우'
}

location_suffixes = ('점', '역', '시장', '사거리', '정거장', '마트', '은행', '도서관', '병원', '의원')
verb_suffixes = (
    '하다', '했다', '한다', '된다', '있다', '없다', '아니다', '아닌', '아니라',
    '하려', '하려고', '하려니', '하는', '하며', '하게', '했어', '했어요',
    '입니다', '됩니다', '같다', '같은', '였다', '했는데', '하는데', '되는데',
    '본', '먹어', '먹은', '먹음', '먹고', '보니', '보면', '보고', '봤'
)


def normalize_entity(text: str) -> str:
    cleaned = re.sub(r"[^0-9A-Za-z가-힣\s]", " ", text)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    tokens = []
    for token in cleaned.split():
        token = re.sub(r"(에서|으로|에게|까지|부터|만|은|는|이|가|을|를|도|에)$", "", token)
        if len(token) <= 1:
            continue
        tokens.append(token)
    return ' '.join(tokens).strip()


def ends_with_verb(text: str) -> bool:
    if text.endswith(verb_suffixes):
        return True
    if text.endswith('다') and len(text) <= 4:
        return True
    return False


def is_valid_entity(text: str) -> bool:
    if len(text) < 2:
        return False
    if text in stopwords:
        return False
    if len(text) > 24:
        return False
    if text.isdigit():
        return False
    tokens = text.split()
    if len(tokens) > 4:
        return False
    if any(tok in stopwords for tok in tokens):
        return False
    if any(tok.endswith(verb_suffixes) or tok in verb_suffixes for tok in tokens):
        return False
    if ends_with_verb(text):
        return False
    if text.endswith(location_suffixes):
        return False
    last = tokens[-1]
    if len(last) == 1 or last in stopwords or last.endswith(verb_suffixes):
        return False
    return True


def detect_regex_entities(text: str):
    patterns = [
        r"[가-힣]+\s*\d+번?",   # 버스 51 / 버스51번 / 노선 3
        r"\d+번\s*[가-힣]+",   # 51번 버스
    ]
    matches = set()
    for pat in patterns:
        for m in re.findall(pat, text):
            matches.add(m.strip())
    return matches


def tag_from_ner(entity_group: str, score: float, threshold: float = 0.70) -> str:
    if score < threshold:
        return 'entity_unknown'
    if entity_group in {'PER', 'ORG', 'LOC', 'MISC'}:
        return entity_group
    return 'entity_unknown'


entity_rows = []
for idx, row in df.iterrows():
    text = str(row['review_text'])
    candidates = set()

    for extractor in (kw_unigram, kw_ngram):
        for phrase, _ in extractor.extract_keywords(text):
            candidates.add((phrase, 'yake', None))

    for phrase in detect_regex_entities(text):
        candidates.add((phrase, 'regex', None))

    try:
        ner_results = ner_model(text)
    except Exception:
        ner_results = []

    for r in ner_results:
        phrase = r.get('word', '')
        score = float(r.get('score', 0.0))
        entity_group = r.get('entity_group', '')
        candidates.add((phrase, 'ner', (entity_group, score)))

    for phrase, source, ner_meta in candidates:
        normalized = normalize_entity(phrase)
        if not normalized or not is_valid_entity(normalized):
            continue
        tag = None
        conf = None
        if source == 'ner' and ner_meta:
            entity_group, score = ner_meta
            tag = tag_from_ner(entity_group, score)
            conf = score
        entity_rows.append({
            'review_id': idx,
            'entity': normalized,
            'entity_source': source,
            'entity_tag': tag or 'entity_unknown',
            'entity_confidence': conf,
        })

entities_df = pd.DataFrame(entity_rows).drop_duplicates()
entities_df.head()


,review_id,entity
0,0,배달하기
1,0,지점중
2,0,주문가
3,0,BHC
4,0,쏘마치


In [4]:
# Sentiment analysis model (multilingual). The model outputs 1-5 stars.

sentiment_model = pipeline(
    'sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    truncation=True,
)


Device set to use cpu


In [11]:
# Build entity-level sentiment from the sentence that mentions the entity

def split_sentences(text: str):
    parts = re.split(r"[.!?\\n]+", text)
    return [p.strip() for p in parts if p.strip()]


def find_context(review_text: str, entity: str) -> str:
    for sent in split_sentences(review_text):
        if entity in sent:
            return sent
    return review_text


mentions_df = entities_df.copy()
mentions_df['review_text'] = mentions_df['review_id'].map(df.set_index('review_id')['review_text'])
mentions_df['context_text'] = mentions_df.apply(
    lambda r: find_context(r['review_text'], r['entity']), axis=1
)


def batch_sentiment(texts, batch_size=32):
    scores = []
    labels = []
    confidences = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        results = sentiment_model(batch)
        for r in results:
            stars = int(r['label'].split()[0])
            scores.append(stars)
            labels.append(r['label'])
            confidences.append(r['score'])
    return scores, labels, confidences


scores, labels, confidences = batch_sentiment(mentions_df['context_text'].tolist())
mentions_df['sentiment_score'] = scores
mentions_df['sentiment_label'] = labels
mentions_df['sentiment_confidence'] = confidences

mentions_df.head()


,review_id,entity,review_text,context_text,sentiment_score,sentiment_label,sentiment_confidence
0,0,배달하기,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문. 배달 받고 보니 서비스라고 콜라까지 보냈다. 따로 리뷰 요청도 없었는데. 마늘 토핑은 안쪽까지 바삭하니 과자 식감이 신기했다.,쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문,3,3 stars,0.227882
1,0,지점중,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문. 배달 받고 보니 서비스라고 콜라까지 보냈다. 따로 리뷰 요청도 없었는데. 마늘 토핑은 안쪽까지 바삭하니 과자 식감이 신기했다.,근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문,1,1 star,0.337694
2,0,주문가,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문. 배달 받고 보니 서비스라고 콜라까지 보냈다. 따로 리뷰 요청도 없었는데. 마늘 토핑은 안쪽까지 바삭하니 과자 식감이 신기했다.,근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문,1,1 star,0.337694
3,0,BHC,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문. 배달 받고 보니 서비스라고 콜라까지 보냈다. 따로 리뷰 요청도 없었는데. 마늘 토핑은 안쪽까지 바삭하니 과자 식감이 신기했다.,BHC서현시범단지점,5,5 stars,0.469422
4,0,쏘마치,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문. 배달 받고 보니 서비스라고 콜라까지 보냈다. 따로 리뷰 요청도 없었는데. 마늘 토핑은 안쪽까지 바삭하니 과자 식감이 신기했다.,쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문,3,3 stars,0.227882


In [12]:
# Cluster similar entity mentions to merge variants

embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

unique_entities = sorted(mentions_df['entity'].unique().tolist())
entity_embeddings = embedder.encode(unique_entities, convert_to_tensor=False, normalize_embeddings=True)

if len(unique_entities) >= 2:
    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric='cosine',
        linkage='average',
        distance_threshold=0.30,
    )
    labels = clustering.fit_predict(entity_embeddings)
else:
    labels = [0] * len(unique_entities)

cluster_map = {entity: f"cluster_{label}" for entity, label in zip(unique_entities, labels)}
mentions_df['entity_cluster'] = mentions_df['entity'].map(cluster_map)

mentions_df.head()


,review_id,entity,review_text,context_text,sentiment_score,sentiment_label,sentiment_confidence,entity_cluster
0,0,배달하기,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문. 배달 받고 보니 서비스라고 콜라까지 보냈다. 따로 리뷰 요청도 없었는데. 마늘 토핑은 안쪽까지 바삭하니 과자 식감이 신기했다.,쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문,3,3 stars,0.227882,cluster_3
1,0,지점중,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문. 배달 받고 보니 서비스라고 콜라까지 보냈다. 따로 리뷰 요청도 없었는데. 마늘 토핑은 안쪽까지 바삭하니 과자 식감이 신기했다.,근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문,1,1 star,0.337694,cluster_1
2,0,주문가,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문. 배달 받고 보니 서비스라고 콜라까지 보냈다. 따로 리뷰 요청도 없었는데. 마늘 토핑은 안쪽까지 바삭하니 과자 식감이 신기했다.,근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문,1,1 star,0.337694,cluster_1
3,0,BHC,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문. 배달 받고 보니 서비스라고 콜라까지 보냈다. 따로 리뷰 요청도 없었는데. 마늘 토핑은 안쪽까지 바삭하니 과자 식감이 신기했다.,BHC서현시범단지점,5,5 stars,0.469422,cluster_21
4,0,쏘마치,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문. 배달 받고 보니 서비스라고 콜라까지 보냈다. 따로 리뷰 요청도 없었는데. 마늘 토핑은 안쪽까지 바삭하니 과자 식감이 신기했다.,쏘마치를 주문하려니 배달하기 천원이 모자라 마늘 토핑 추가 주문,3,3 stars,0.227882,cluster_1


In [13]:
# Aggregate sentiment per entity and keep cluster id for grouping

entity_stats = (
    mentions_df
    .groupby('entity')
    .agg(
        entity_cluster=('entity_cluster', lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
        entity_tag=('entity_tag', lambda x: x.mode().iloc[0] if len(x.mode()) else 'entity_unknown'),
        n_mentions=('entity', 'count'),
        n_reviews=('review_id', 'nunique'),
        mean_sentiment=('sentiment_score', 'mean'),
        median_sentiment=('sentiment_score', 'median'),
    )
    .reset_index()
    .sort_values(by=['mean_sentiment', 'n_mentions'], ascending=[False, False])
)

cluster_stats = (
    mentions_df
    .groupby('entity_cluster')
    .agg(
        n_mentions=('entity', 'count'),
        unique_entities=('entity', lambda x: sorted(set(x))),
        mean_sentiment=('sentiment_score', 'mean'),
        median_sentiment=('sentiment_score', 'median'),
    )
    .reset_index()
    .sort_values(by='mean_sentiment', ascending=False)
)

entity_stats


,entity,entity_cluster,n_mentions,n_reviews,mean_sentiment,median_sentiment
0,BHC,cluster_21,1,1,5.0,5.0
24,고추바사삭,cluster_1,1,1,5.0,5.0
27,광고,cluster_1,1,1,5.0,5.0
32,구글맵,cluster_31,1,1,5.0,5.0
34,굽네치킨,cluster_1,1,1,5.0,5.0
...,...,...,...,...,...,...
326,하나 받으러 오라고,cluster_1,1,1,1.0,1.0
330,한셋트,cluster_7,1,1,1.0,1.0
331,한셋트 치약이었다,cluster_1,1,1,1.0,1.0
338,할인한다기 주문하려했더니,cluster_1,1,1,1.0,1.0


In [15]:
df = entity_stats
df

,entity,entity_cluster,n_mentions,n_reviews,mean_sentiment,median_sentiment
0,BHC,cluster_21,1,1,5.0,5.0
24,고추바사삭,cluster_1,1,1,5.0,5.0
27,광고,cluster_1,1,1,5.0,5.0
32,구글맵,cluster_31,1,1,5.0,5.0
34,굽네치킨,cluster_1,1,1,5.0,5.0
...,...,...,...,...,...,...
326,하나 받으러 오라고,cluster_1,1,1,1.0,1.0
330,한셋트,cluster_7,1,1,1.0,1.0
331,한셋트 치약이었다,cluster_1,1,1,1.0,1.0
338,할인한다기 주문하려했더니,cluster_1,1,1,1.0,1.0


In [22]:
df.groupby('entity_cluster')['entity'].count().sort_values()[-5:]

entity_cluster
cluster_9       5
cluster_16      7
cluster_3       8
cluster_22     16
cluster_1     240
Name: entity, dtype: int64

In [32]:
df[df['entity_cluster'] == 'cluster_5']

,entity,entity_cluster,n_mentions,n_reviews,mean_sentiment,median_sentiment
138,발뒤꿈치,cluster_5,2,2,2.0,2.0
63,니자 발뒤꿈치,cluster_5,1,1,2.0,2.0
137,발가락 아파서,cluster_5,1,1,2.0,2.0
136,발가락,cluster_5,1,1,1.0,1.0


In [33]:
df[df['entity']=='정형외과']

,entity,entity_cluster,n_mentions,n_reviews,mean_sentiment,median_sentiment
266,정형외과,cluster_1,3,3,1.666667,1.0


In [35]:
df.sort_values('mean_sentiment', ascending=False)[:10]

,entity,entity_cluster,n_mentions,n_reviews,mean_sentiment,median_sentiment
0,BHC,cluster_21,1,1,5.0,5.0
24,고추바사삭,cluster_1,1,1,5.0,5.0
27,광고,cluster_1,1,1,5.0,5.0
32,구글맵,cluster_31,1,1,5.0,5.0
34,굽네치킨,cluster_1,1,1,5.0,5.0
51,깐풍치킨중,cluster_1,1,1,5.0,5.0
114,명성왕족발,cluster_4,1,1,5.0,5.0
124,민생지원금,cluster_28,1,1,5.0,5.0
184,소스 느낌,cluster_10,1,1,5.0,5.0
255,자담치킨,cluster_1,1,1,5.0,5.0


In [10]:
entity_stats.to_csv('entity_stats.csv', index=False)


## Next steps
- Improve entity extraction with language-specific tokenizers or NER models if the data distribution is known.
- Move from review-level to entity-level sentiment by pairing entity spans with sentence-level sentiment.
- Experiment with different clustering thresholds or algorithms (e.g., HDBSCAN) to better control entity merging.
- Persist intermediate artifacts (embeddings, clusters) for incremental updates on new reviews.
